Exemple complet PyTorch : prévision de la consommation électrique à partir de
Température, Humidité, Ensoleillement

In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------------
# 1. Génération de données multivariées
# -----------------------------------
np.random.seed(0)
n = 1500
t = np.arange(n)

temperature = 15 + 10*np.sin(t/200) + np.random.randn(n)
humidity = 50 + 15*np.cos(t/300) + np.random.randn(n)
sun = np.maximum(0, 10*np.sin(t/100))  # ensoleillement >= 0

# consommation influencée par les 3 facteurs
consumption = (
    100 
    + 2*temperature 
    - 0.5*humidity 
    + 3*sun 
    + 5*np.random.randn(n)
)

# X = variables explicatives, y = cible
X = np.column_stack((temperature, humidity, sun))
y = consumption.reshape(-1, 1)

# Normalisation
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# -----------------------------------
# 2. Création des séquences temporelles
# -----------------------------------
def create_sequences(X, y, seq_length):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:i+seq_length, :])  # (L, 3)
        ys.append(y[i+seq_length])
    return np.array(Xs), np.array(ys)

seq_length = 24  # ex: 24 heures
Xs, ys = create_sequences(X, y, seq_length)

# Train / Test
train_size = int(0.8 * len(Xs))
X_train, X_test = Xs[:train_size], Xs[train_size:]
y_train, y_test = ys[:train_size], ys[train_size:]

# Tenseurs PyTorch
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

print("Forme des données :", X_train.shape)
# (batch, 24, 3)

# -----------------------------------
# 3. Modèle LSTM
# -----------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]     # dernier pas de temps
        out = self.fc(out)
        return out

# -----------------------------------
# 4. Modèle GRU
# -----------------------------------
class GRUModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# -----------------------------------
# 5. Entraînement générique
# -----------------------------------
def train_model(model, X_train, y_train, epochs=25, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        if (epoch+1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# -----------------------------------
# 6. Évaluation
# -----------------------------------
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(X_test).numpy()
        y_true = y_test.numpy()

    # retour à l’échelle réelle
    preds = scaler_y.inverse_transform(preds)
    y_true = scaler_y.inverse_transform(y_true)

    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100
    return rmse, mae, mape

# -----------------------------------
# 7. LSTM
# -----------------------------------
print("===== LSTM multivarié =====")
lstm = LSTMModel(input_size=3)
train_model(lstm, X_train, y_train)

rmse, mae, mape = evaluate_model(lstm, X_test, y_test)
print(f"LSTM -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, MAPE: {mape:.2f}%")

# -----------------------------------
# 8. GRU
# -----------------------------------
print("\n===== GRU multivarié =====")
gru = GRUModel(input_size=3)
train_model(gru, X_train, y_train)

rmse, mae, mape = evaluate_model(gru, X_test, y_test)
print(f"GRU -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, MAPE: {mape:.2f}%")


Forme des données : torch.Size([1180, 24, 3])
===== LSTM multivarié =====
Epoch [5/25], Loss: 0.0694
Epoch [10/25], Loss: 0.0249
Epoch [15/25], Loss: 0.0305
Epoch [20/25], Loss: 0.0204
Epoch [25/25], Loss: 0.0191
LSTM -> RMSE: 19.45, MAE: 16.84, MAPE: 12.00%

===== GRU multivarié =====
Epoch [5/25], Loss: 0.0314
Epoch [10/25], Loss: 0.0331
Epoch [15/25], Loss: 0.0141
Epoch [20/25], Loss: 0.0121
Epoch [25/25], Loss: 0.0113
GRU -> RMSE: 15.53, MAE: 13.58, MAPE: 9.71%
